# FitoDiag-PY — Entrenamiento en Google Colab

Este notebook corre todo el pipeline de datos y entrenamiento en una GPU gratuita de Colab.

**Antes de empezar:** `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)`.

Tiempo aproximado con T4: preparación 10 min, cada modelo 15 a 40 min.

## 1. Clonar el repo y montar Drive (para no perder los modelos si se corta la sesión)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/USUARIO/fitodiag-py.git'  # <- cambiar por la URL de su repo
!git clone $REPO /content/fitodiag-py
%cd /content/fitodiag-py
!pip install -q -r requirements/train.txt

## 2. Descargar los datasets

1. En kaggle.com: *Settings > API > Create New Token* (baja `kaggle.json`).
2. Aceptar las reglas en https://www.kaggle.com/competitions/cassava-leaf-disease-classification/rules
3. Subir `kaggle.json` con la celda de abajo.
4. El sésamo se baja a mano desde Mendeley y se sube a Drive (ver paso 2b).

In [ ]:
from google.colab import files
import os
os.makedirs('/root/.kaggle', exist_ok=True)
up = files.upload()  # elegir kaggle.json
!mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

### 2b. Sésamo: descomprimir el ZIP de Mendeley desde Drive

In [ ]:
SESAMO_ZIP = '/content/drive/MyDrive/fitodiag/sesame.zip'  # <- ruta donde subieron el ZIP
!mkdir -p data/raw/sesamo && unzip -q -o "$SESAMO_ZIP" -d data/raw/sesamo

In [ ]:
!bash scripts/download_data.sh

## 3. Preparar el dataset unificado

In [ ]:
!python scripts/prepare_data.py --raw data/raw --out data/processed --max-per-class 1500
!cat data/processed/stats.json | head -40

## 4. Entrenar los dos modelos y la ablación

In [ ]:
OUT = '/content/drive/MyDrive/fitodiag/models'   # se guarda en Drive
!python scripts/train.py --arch efficientnet_b0 --epochs 15 --out $OUT

In [ ]:
!python scripts/train.py --arch resnet50 --epochs 15 --out $OUT

In [ ]:
# Ablación: mismo EfficientNet-B0 sin data augmentation ni sampler ponderado.
# Muestra cuánto aportan las técnicas contra el desbalance (va al informe).
!python scripts/train.py --arch efficientnet_b0 --run-name effnet_baseline --no-augment --no-balance --epochs 15 --out $OUT

## 5. Evaluar en test y generar tablas/figuras del informe

In [ ]:
for run in ['efficientnet_b0', 'resnet50', 'effnet_baseline']:
    !python scripts/evaluate.py --run $OUT/$run
!python scripts/compare_runs.py --runs $OUT/efficientnet_b0 $OUT/resnet50 $OUT/effnet_baseline --best $OUT/efficientnet_b0
!cat results.md

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT}/efficientnet_b0/confusion_matrix.png'))

## 6. Descargar lo que va al repo

- `models/best_model.pt` (el modelo elegido, para la API)
- `paper/tables/*.tex` y `paper/figures/*.pdf` (para el informe)
- `results.md` (para el README)

In [ ]:
!cp $OUT/efficientnet_b0/best_model.pt models/best_model.pt
!zip -r -q /content/entregables.zip models/best_model.pt paper/tables paper/figures results.md data/README.md
files.download('/content/entregables.zip')